# RDMA Fundamentals

Establish a safe, read-only baseline for RDMA device and utility discovery.

## Objectives

Identify available RDMA tools and frame an experiment around devices, ports, and data movement.

## Background

RDMA enables direct data movement with low CPU involvement, but measurements depend on device, link, memory registration, queue, and topology details.

## Prediction

The DGX Spark should expose one or more Mellanox/NVIDIA RDMA-capable devices through the Linux RDMA subsystem.

Before running the experiment, I predict:

1. At least one of `ibv_devices`, `ibstat`, and `rdma` will be installed.
2. `/sys/class/infiniband` will contain one or more RDMA devices associated with the high-speed ConnectX interfaces.
3. Each physical port will report a link layer, administrative state, physical state, and nominal link rate.
4. The RDMA devices will map to Linux network interfaces, but an active physical link does not necessarily imply that an IP address or usable RDMA route is configured.
5. The reported link layer may be Ethernet rather than native InfiniBand. In that case, later RDMA experiments would use RoCE and would depend on Ethernet addressing and configuration.

These are architectural expectations, not measured properties. The experiment below records the actual local configuration.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
from common.rdma import detect_rdma_utilities, run_rdma_utility


rdma_utilities = detect_rdma_utilities()

for utility in rdma_utilities:
    status = "available" if utility.available else "missing"
    print(f"{utility.name:<12} {status:<9} {utility.path or '-'}")

ibv_devices  available /usr/bin/ibv_devices
ibstat       missing   -
rdma         available /usr/bin/rdma


In [3]:
def print_command_result(result) -> None:
    command_text = " ".join(result.command)

    print(f"$ {command_text}")
    print(f"return code: {result.returncode}")

    if result.timed_out:
        print("status: timed out")
    elif result.executable_missing:
        print("status: executable missing")
    elif result.error is not None:
        print(f"status: {result.error}")
    else:
        print(f"status: {'succeeded' if result.succeeded else 'failed'}")

    if result.stdout.strip():
        print("\nstdout:")
        print(result.stdout.rstrip())

    if result.stderr.strip():
        print("\nstderr:")
        print(result.stderr.rstrip())

    print()


available_utility_names = {
    utility.name for utility in rdma_utilities if utility.available
}

rdma_command_results = []

if "ibv_devices" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibv_devices"))

if "ibstat" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibstat"))

if "rdma" in available_utility_names:
    rdma_command_results.extend(
        [
            run_rdma_utility("rdma", "dev", "show"),
            run_rdma_utility("rdma", "link", "show"),
        ]
    )

if not rdma_command_results:
    print("No supported RDMA utilities were available.")
else:
    for result in rdma_command_results:
        print_command_result(result)

$ ibv_devices
return code: 0
status: succeeded

stdout:
    device          	   node GUID
    ------          	----------------
    rocep1s0f0      	4cbb470300830241
    rocep1s0f1      	4cbb470300830242
    roceP2p1s0f0    	4cbb470300830245
    roceP2p1s0f1    	4cbb470300830246

$ rdma dev show
return code: 0
status: succeeded

stdout:
0: rocep1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0241 sys_image_guid 4cbb:4703:0083:0241 
1: rocep1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0242 sys_image_guid 4cbb:4703:0083:0241 
2: roceP2p1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0245 sys_image_guid 4cbb:4703:0083:0241 
3: roceP2p1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0246 sys_image_guid 4cbb:4703:0083:0241

$ rdma link show
return code: 0
status: succeeded

stdout:
link rocep1s0f0/1 state DOWN physical_state DISABLED netdev enp1s0f0np0 
link rocep1s0f1/1 state ACTIVE physical_state LINK_UP netdev enp1s0f1np1 
link roceP2p1s0f0/1

### Linux sysfs inventory

Command-line utilities present a user-space view of the RDMA subsystem. Linux also exposes device, port, and network-interface relationships through `/sys/class/infiniband`.

The following cell reads that hierarchy directly so that the inventory remains useful even if some optional RDMA utilities are absent.

In [4]:
import pandas as pd


def read_text(path: Path) -> str | None:
    try:
        return path.read_text().strip()
    except (FileNotFoundError, PermissionError, OSError):
        return None


infiniband_root = Path("/sys/class/infiniband")
rdma_device_rows = []
rdma_port_rows = []

if not infiniband_root.is_dir():
    print(f"{infiniband_root} does not exist.")
else:
    rdma_devices = sorted(path for path in infiniband_root.iterdir() if path.is_dir())

    if not rdma_devices:
        print(f"No RDMA devices were found under {infiniband_root}.")

    for rdma_device in rdma_devices:
        device_path = rdma_device / "device"
        network_path = device_path / "net"

        network_interfaces = (
            sorted(path.name for path in network_path.iterdir())
            if network_path.is_dir()
            else []
        )

        pci_device = None
        try:
            pci_device = device_path.resolve().name
        except OSError:
            pass

        driver = None
        driver_path = device_path / "driver"
        try:
            if driver_path.exists():
                driver = driver_path.resolve().name
        except OSError:
            pass

        rdma_device_rows.append(
            {
                "rdma_device": rdma_device.name,
                "node_type": read_text(rdma_device / "node_type"),
                "firmware_version": read_text(rdma_device / "fw_ver"),
                "node_guid": read_text(rdma_device / "node_guid"),
                "sys_image_guid": read_text(rdma_device / "sys_image_guid"),
                "pci_device": pci_device,
                "driver": driver,
                "network_interfaces": ", ".join(network_interfaces) or None,
            }
        )

        ports_path = rdma_device / "ports"
        if not ports_path.is_dir():
            continue

        for port_path in sorted(
            ports_path.iterdir(),
            key=lambda path: int(path.name) if path.name.isdigit() else path.name,
        ):
            if not port_path.is_dir():
                continue

            rdma_port_rows.append(
                {
                    "rdma_device": rdma_device.name,
                    "port": port_path.name,
                    "state": read_text(port_path / "state"),
                    "physical_state": read_text(port_path / "phys_state"),
                    "link_layer": read_text(port_path / "link_layer"),
                    "rate": read_text(port_path / "rate"),
                    "lid": read_text(port_path / "lid"),
                    "lid_mask_count": read_text(port_path / "lid_mask_count"),
                    "sm_lid": read_text(port_path / "sm_lid"),
                }
            )

rdma_devices_df = pd.DataFrame(rdma_device_rows)
rdma_ports_df = pd.DataFrame(rdma_port_rows)

print("RDMA devices")
display(rdma_devices_df)

print("\nRDMA ports")
display(rdma_ports_df)

RDMA devices


,rdma_device,node_type,firmware_version,node_guid,sys_image_guid,pci_device,driver,network_interfaces
0,roceP2p1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0245,4cbb:4703:0083:0241,0002:01:00.0,mlx5_core,enP2p1s0f0np0
1,roceP2p1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0246,4cbb:4703:0083:0241,0002:01:00.1,mlx5_core,enP2p1s0f1np1
2,rocep1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0241,4cbb:4703:0083:0241,0000:01:00.0,mlx5_core,enp1s0f0np0
3,rocep1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0242,4cbb:4703:0083:0241,0000:01:00.1,mlx5_core,enp1s0f1np1



RDMA ports


,rdma_device,port,state,physical_state,link_layer,rate,lid,lid_mask_count,sm_lid
0,roceP2p1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
1,roceP2p1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0
2,rocep1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
3,rocep1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0


### Associated network interfaces

RDMA device discovery alone does not show whether the corresponding Linux network interfaces are administratively enabled, physically connected, or assigned addresses.

The next cell reads the ordinary Linux interface state and uses `ip` only for read-only address and route reporting.

In [5]:
from common.utils import run_command


rdma_network_interfaces = sorted(
    {
        interface
        for row in rdma_device_rows
        for interface in (row["network_interfaces"] or "").split(", ")
        if interface
    }
)

network_interface_rows = []

for interface in rdma_network_interfaces:
    interface_path = Path("/sys/class/net") / interface

    network_interface_rows.append(
        {
            "interface": interface,
            "operstate": read_text(interface_path / "operstate"),
            "carrier": read_text(interface_path / "carrier"),
            "mtu": read_text(interface_path / "mtu"),
            "speed_mbps": read_text(interface_path / "speed"),
            "duplex": read_text(interface_path / "duplex"),
            "address": read_text(interface_path / "address"),
        }
    )

network_interfaces_df = pd.DataFrame(network_interface_rows)

print("RDMA-associated network interfaces")
display(network_interfaces_df)

if not rdma_network_interfaces:
    print("No Linux network interfaces were mapped to the RDMA devices.")
else:
    for interface in rdma_network_interfaces:
        print_command_result(
            run_command(("ip", "-details", "address", "show", "dev", interface))
        )

    print_command_result(run_command(("ip", "route", "show")))

RDMA-associated network interfaces


,interface,operstate,carrier,mtu,speed_mbps,duplex,address
0,enP2p1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:45
1,enP2p1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:46
2,enp1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:41
3,enp1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:42


$ ip -details address show dev enP2p1s0f0np0
return code: 0
status: succeeded

stdout:
5: enP2p1s0f0np0: <NO-CARRIER,BROADCAST,MULTICAST,UP> mtu 1500 qdisc mq state DOWN group default qlen 1000
    link/ether 4c:bb:47:83:02:45 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p0 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.0

$ ip -details address show dev enP2p1s0f1np1
return code: 0
status: succeeded

stdout:
6: enP2p1s0f1np1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc mq state UP group default qlen 1000
    link/ether 4c:bb:47:83:02:46 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p1 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.1 
    

### Prediction: peer reachability and benchmark support

Both active `/30` networks should contain one remote endpoint, conventionally `.2`:

- local `10.200.0.1` should reach peer `10.200.0.2` through `enP2p1s0f1np1`;
- local `10.201.0.1` should reach peer `10.201.0.2` through `enp1s0f1np1`.

Linux route lookup should select a different physical interface for each peer.

The system may also provide additional verbs and `perftest` utilities beyond the three commands currently handled by `common.rdma`. Discovering those utilities does not prove that RDMA communication works; it only determines which later experiments are locally available.

In [6]:
from shutil import which


candidate_rdma_utilities = (
    "ibv_devinfo",
    "ibv_rc_pingpong",
    "rping",
    "ib_send_lat",
    "ib_write_lat",
    "ib_read_lat",
    "ib_send_bw",
    "ib_write_bw",
    "ib_read_bw",
)

rdma_tool_rows = [
    {
        "utility": utility,
        "available": which(utility) is not None,
        "path": which(utility),
    }
    for utility in candidate_rdma_utilities
]

rdma_tools_df = pd.DataFrame(rdma_tool_rows)
display(rdma_tools_df)

,utility,available,path
0,ibv_devinfo,True,/usr/bin/ibv_devinfo
1,ibv_rc_pingpong,True,/usr/bin/ibv_rc_pingpong
2,rping,True,/usr/bin/rping
3,ib_send_lat,True,/usr/bin/ib_send_lat
4,ib_write_lat,True,/usr/bin/ib_write_lat
5,ib_read_lat,True,/usr/bin/ib_read_lat
6,ib_send_bw,True,/usr/bin/ib_send_bw
7,ib_write_bw,True,/usr/bin/ib_write_bw
8,ib_read_bw,True,/usr/bin/ib_read_bw


In [7]:
peer_paths = (
    {
        "local_address": "10.200.0.1",
        "peer_address": "10.200.0.2",
        "expected_interface": "enP2p1s0f1np1",
    },
    {
        "local_address": "10.201.0.1",
        "peer_address": "10.201.0.2",
        "expected_interface": "enp1s0f1np1",
    },
)

peer_path_results = []

for path in peer_paths:
    route_result = run_command(
        ("ip", "route", "get", path["peer_address"]),
        timeout=5.0,
    )
    ping_result = run_command(
        (
            "ping",
            "-n",
            "-c",
            "5",
            "-W",
            "1",
            "-I",
            path["expected_interface"],
            path["peer_address"],
        ),
        timeout=10.0,
    )

    peer_path_results.append(
        {
            **path,
            "route_succeeded": route_result.succeeded,
            "route_output": route_result.stdout.strip(),
            "ping_succeeded": ping_result.succeeded,
            "ping_output": ping_result.stdout.strip(),
            "ping_error": ping_result.stderr.strip() or ping_result.error,
        }
    )

    print_command_result(route_result)
    print_command_result(ping_result)

peer_paths_df = pd.DataFrame(peer_path_results)
display(
    peer_paths_df[
        [
            "local_address",
            "peer_address",
            "expected_interface",
            "route_succeeded",
            "ping_succeeded",
        ]
    ]
)

$ ip route get 10.200.0.2
return code: 0
status: succeeded

stdout:
10.200.0.2 dev enP2p1s0f1np1 src 10.200.0.1 uid 1001 
    cache

$ ping -n -c 5 -W 1 -I enP2p1s0f1np1 10.200.0.2
return code: 0
status: succeeded

stdout:
PING 10.200.0.2 (10.200.0.2) from 10.200.0.1 enP2p1s0f1np1: 56(84) bytes of data.
64 bytes from 10.200.0.2: icmp_seq=1 ttl=64 time=0.404 ms
64 bytes from 10.200.0.2: icmp_seq=2 ttl=64 time=0.749 ms
64 bytes from 10.200.0.2: icmp_seq=3 ttl=64 time=0.703 ms
64 bytes from 10.200.0.2: icmp_seq=4 ttl=64 time=0.339 ms
64 bytes from 10.200.0.2: icmp_seq=5 ttl=64 time=0.732 ms

--- 10.200.0.2 ping statistics ---
5 packets transmitted, 5 received, 0% packet loss, time 4134ms
rtt min/avg/max/mdev = 0.339/0.585/0.749/0.176 ms

$ ip route get 10.201.0.2
return code: 0
status: succeeded

stdout:
10.201.0.2 dev enp1s0f1np1 src 10.201.0.1 uid 1001 
    cache

$ ping -n -c 5 -W 1 -I enp1s0f1np1 10.201.0.2
return code: 0
status: succeeded

stdout:
PING 10.201.0.2 (10.201.0.2) from 10

,local_address,peer_address,expected_interface,route_succeeded,ping_succeeded
0,10.200.0.1,10.200.0.2,enP2p1s0f1np1,True,True
1,10.201.0.1,10.201.0.2,enp1s0f1np1,True,True


In [8]:
if which("ibv_devinfo") is None:
    print("ibv_devinfo is not installed.")
else:
    ibv_devinfo_result = run_command(("ibv_devinfo", "-v"), timeout=30.0)
    print_command_result(ibv_devinfo_result)

$ ibv_devinfo -v
return code: 0
status: succeeded

stdout:
hca_id:	rocep1s0f0
	transport:			InfiniBand (0)
	fw_ver:				28.45.4028
	node_guid:			4cbb:4703:0083:0241
	sys_image_guid:			4cbb:4703:0083:0241
	vendor_id:			0x02c9
	vendor_part_id:			4129
	hw_ver:				0x0
	board_id:			NVD0000000087
	phys_port_cnt:			1
	max_mr_size:			0xffffffffffffffff
	page_size_cap:			0xfffffffffffff000
	max_qp:				131072
	max_qp_wr:			8192
	device_cap_flags:		0x25321c36
					BAD_PKEY_CNTR
					BAD_QKEY_CNTR
					AUTO_PATH_MIG
					CHANGE_PHY_PORT
					PORT_ACTIVE_EVENT
					SYS_IMAGE_GUID
					RC_RNR_NAK_GEN
					MEM_WINDOW
					XRC
					MEM_MGT_EXTENSIONS
					MEM_WINDOW_TYPE_2B
					RAW_IP_CSUM
					MANAGED_FLOW_STEERING
	max_sge:			30
	max_sge_rd:			30
	max_cq:				16777216
	max_cqe:			4194303
	max_mr:				16777216
	max_pd:				8388608
	max_qp_rd_atom:			16
	max_ee_rd_atom:			0
	max_res_rd_atom:		2097152
	max_qp_init_rd_atom:		16
	max_ee_init_rd_atom:		0
	atomic_cap:			ATOMIC_HCA (1)
	max_ee:				0
	max_rdd:	

### Prediction: reliable-connected verbs communication

The `10.200.0.0/30` path is reachable over the active `roceP2p1s0f1` adapter, and GID index 3 represents its IPv4 RoCE v2 address.

I predict that:

1. the peer exposes a matching active RDMA adapter and IPv4 RoCE v2 GID;
2. a reliable-connected queue pair can transition to the ready states on both hosts;
3. `ibv_rc_pingpong` can exchange messages across the direct link;
4. the test will report successful iterations without packet loss or retry failure.

This is a functional smoke test. Its reported throughput and latency are not yet treated as controlled benchmark results.

In [9]:
peer_host = "spark-f868"

remote_inventory_commands = (
    ("hostname",),
    ("ip", "-brief", "address", "show", "dev", "enP2p1s0f1np1"),
    ("rdma", "link", "show", "roceP2p1s0f1/1"),
    ("ibv_devinfo", "-d", "roceP2p1s0f1"),
)

remote_inventory_results = []

for remote_command in remote_inventory_commands:
    result = run_command(
        (
            "ssh",
            "-o",
            "BatchMode=yes",
            "-o",
            "ConnectTimeout=5",
            peer_host,
            *remote_command,
        ),
        timeout=15.0,
    )
    remote_inventory_results.append(result)
    print_command_result(result)

if not all(result.succeeded for result in remote_inventory_results):
    raise RuntimeError("Remote inventory failed; do not start the verbs experiment.")

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 hostname
return code: 0
status: succeeded

stdout:
spark-f868

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 ip -brief address show dev enP2p1s0f1np1
return code: 0
status: succeeded

stdout:
enP2p1s0f1np1    UP             10.200.0.2/30

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 rdma link show roceP2p1s0f1/1
return code: 0
status: succeeded

stdout:
link roceP2p1s0f1/1 state ACTIVE physical_state LINK_UP netdev enP2p1s0f1np1

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 ibv_devinfo -d roceP2p1s0f1
return code: 0
status: succeeded

stdout:
hca_id:	roceP2p1s0f1
	transport:			InfiniBand (0)
	fw_ver:				28.45.4028
	node_guid:			4cbb:4703:0082:f86e
	sys_image_guid:			4cbb:4703:0082:f869
	vendor_id:			0x02c9
	vendor_part_id:			4129
	hw_ver:				0x0
	board_id:			NVD0000000087
	phys_port_cnt:			1
		port:	1
			state:			PORT_ACTIVE (4)
			max_mtu:		4096 (5)
			active_mtu:		4096 (5)
			sm_lid:			0
			port_lid:		0
	

### Reliable-connected ping-pong

`ibv_rc_pingpong` creates registered memory, completion queues, and a reliable-connected queue pair on each host. The server waits for a connection; the client connects through the peer's IPv4 address and exchanges a small number of messages.

The SSH process is retained as the server-process handle so that its stdout, stderr, exit status, and cleanup can all be recorded.

In [10]:
import subprocess
import time


rdma_device = "roceP2p1s0f1"
rdma_port = "1"
gid_index = "3"
peer_address = "10.200.0.2"

server_command = (
    "ssh",
    "-o",
    "BatchMode=yes",
    "-o",
    "ConnectTimeout=5",
    peer_host,
    "timeout",
    "20s",
    "ibv_rc_pingpong",
    "-d",
    rdma_device,
    "-i",
    rdma_port,
    "-g",
    gid_index,
)

client_command = (
    "timeout",
    "15s",
    "ibv_rc_pingpong",
    "-d",
    rdma_device,
    "-i",
    rdma_port,
    "-g",
    gid_index,
    peer_address,
)

print("$ " + " ".join(server_command))
server_process = subprocess.Popen(
    server_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    time.sleep(1.0)

    client_result = run_command(client_command, timeout=20.0)
    print_command_result(client_result)

    try:
        server_stdout, server_stderr = server_process.communicate(timeout=10.0)
    except subprocess.TimeoutExpired:
        server_process.terminate()
        try:
            server_stdout, server_stderr = server_process.communicate(timeout=5.0)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_stdout, server_stderr = server_process.communicate()

    server_returncode = server_process.returncode

finally:
    if server_process.poll() is None:
        server_process.terminate()
        try:
            server_process.wait(timeout=5.0)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait()

print("Remote server result")
print(f"return code: {server_returncode}")

if server_stdout.strip():
    print("\nstdout:")
    print(server_stdout.rstrip())

if server_stderr.strip():
    print("\nstderr:")
    print(server_stderr.rstrip())

rc_pingpong_succeeded = client_result.succeeded and server_returncode == 0

print(f"\nRC ping-pong succeeded: {rc_pingpong_succeeded}")

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 timeout 20s ibv_rc_pingpong -d roceP2p1s0f1 -i 1 -g 3
$ timeout 15s ibv_rc_pingpong -d roceP2p1s0f1 -i 1 -g 3 10.200.0.2
return code: 0
status: succeeded

stdout:
  local address:  LID 0x0000, QPN 0x0008ae, PSN 0x782e01, GID ::ffff:10.200.0.1
  remote address: LID 0x0000, QPN 0x0008ae, PSN 0x0a7a58, GID ::ffff:10.200.0.2
8192000 bytes in 0.01 seconds = 7864.63 Mbit/sec
1000 iters in 0.01 seconds = 8.33 usec/iter

Remote server result
return code: 0

stdout:
  local address:  LID 0x0000, QPN 0x0008ae, PSN 0x0a7a58, GID ::ffff:10.200.0.2
  remote address: LID 0x0000, QPN 0x0008ae, PSN 0x782e01, GID ::ffff:10.200.0.1
8192000 bytes in 0.01 seconds = 6845.92 Mbit/sec
1000 iters in 0.01 seconds = 9.57 usec/iter

RC ping-pong succeeded: True


In [11]:
rc_pingpong_summary_df = pd.DataFrame(
    [
        {
            "local_host": socket.gethostname(),
            "peer_host": peer_host,
            "rdma_device": rdma_device,
            "port": int(rdma_port),
            "gid_index": int(gid_index),
            "peer_address": peer_address,
            "client_returncode": client_result.returncode,
            "server_returncode": server_returncode,
            "succeeded": rc_pingpong_succeeded,
        }
    ]
)

display(rc_pingpong_summary_df)

,local_host,peer_host,rdma_device,port,gid_index,peer_address,client_returncode,server_returncode,succeeded
0,spark-0240,spark-f868,roceP2p1s0f1,1,3,10.200.0.2,0,0,True


## Observations

Run the cells above on one DGX Spark and preserve their outputs before drawing conclusions.

The initial interpretation should answer:

1. Which RDMA utilities are installed?
2. Which RDMA devices are visible?
3. Which kernel driver and PCI device back each RDMA device?
4. Which Linux network interfaces map to those devices?
5. What link layer does each RDMA port report?
6. Which ports are active, and what nominal rate do they report?
7. Are the mapped network interfaces up and carrying an IP address?
8. Does the routing table contain routes through those interfaces?

No bandwidth, latency, CPU-overhead, or GPU-direct conclusions can be made from this inventory alone.

## Explanation

TODO: Relate the observed device and link information to the RDMA data path.

## Connection to LLMs

RDMA supports low-latency transfer of tensors and collective traffic in distributed inference and training.

## Further Exploration

TODO: Define a two-node bandwidth and latency experiment with explicit safety and cleanup steps.